<a href="https://colab.research.google.com/github/anandharidas/InferenceEngineering/blob/main/MatMul_Kernels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!ncu --set full -f -o matmul-01.ncu-rep python matmul_script.py

==PROF== Connected to process 5417 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 50 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 50 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 50 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 51 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 50 passes
==PROF== Disconnected from process 5417
==PROF== Report: /content/matmul-01.ncu-rep


In [9]:
%%writefile matmul_script.py
from numba import cuda
import cupy as cp
import numpy as np

ceil_div = lambda a,b : (a+b-1) // b

@cuda.jit
def matmul (matA, matB, matC, m , n , k):
  out_row = cuda.threadIdx.x + cuda.blockDim.x * cuda.blockIdx.x
  out_col = cuda.threadIdx.y + cuda.blockDim.y * cuda.blockIdx.y

  if out_col >= n or out_row >= m:
    return

  dot_product = np.float32(0.0)
  for i in range(k):
    dot_product += matA[out_row,i]* matB[i,out_col]
  matC[out_row,out_col] = dot_product

#matrix dimensions
m = 1000
k = 1200
n = 800

matA= cp.ones(shape = (m,k), dtype = np.float32) * 2.0
matB= cp.ones(shape = (k,n), dtype = np.float32) * 3.0
matC= cp.zeros(shape = (m,n), dtype = np.float32)

threads_per_block = (32,32)
blocks_per_thread = (
    ceil_div (m, threads_per_block[0]),
    ceil_div (n, threads_per_block[1])
)

matmul[blocks_per_thread, threads_per_block](matA, matB, matC, m, n, k)

# Wait for GPU execution to complete before finishing
cuda.synchronize()


Overwriting matmul_script.py
